# Experiment: Plantellect CNN Training

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from sklearn.metrics import confusion_matrix, classification_report
import random
from google.colab import drive

# Settings
Models, image size(224 x 224 pixels) & rgb (3), 


In [ ]:
SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.keras.utils.set_random_seed(SEED)

MODEL_ARCH = "EfficientNetB0"
INPUT_SHAPE = (224, 224, 3)
BATCH_SIZE = 32
EPOCHS = 50
LEARNING_RATE = 1e-4
DROPOUT = 0.5

AUGMENTATION = { "rotation": 20, "zoom": 0.2, "horizontal_flip": True, "brightness": 0.2, "contrast": 0.2}

#Callbacks or checkpoint if the model accuracy degrade
EARLY_STOPPING_PATIENCE = 10
REDUCE_LR_PATIENCE = 5
MODEL_FILENAME = "plantellect_model.keras"

# LOADING THE DATA

In [ ]:
drive.mount('/content/drive')
DATA_DIR = "/content/drive/MyDrive/plantellect/plantellect_training"
OUT_DIR = "/content//drive/Mydrive/plantellect/model/main"
MODEL_PATH = os.path.join(OUT_DIR, MODEL_FILENAME)


train_dir = os.path.join(DATA_DIR, "train")
val_dir   = os.path.join(DATA_DIR, "val")
test_dir  = os.path.join(DATA_DIR, "test")

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".ppm", ".tif", ".tiff")

def class_to_scientific(folder: str) -> str:
    """'Ocotea_acutifolia' or 'Ocotea  acutifolia' -> 'Ocotea acutifolia'."""
    return "_".join(folder.split()).replace("_", " ").strip()


def list_classes(root: str):
    """Directories only — a stray .DS_Store would become a phantom class."""
    return sorted(d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d)))


def count_images(directory: str) -> int:
    """Counts what DirectoryIterator will actually load, recursing like it does."""
    return sum(1 for _, _, files in os.walk(directory)
                 for f in files if f.lower().endswith(IMG_EXT))


class_names = list_classes(train_dir)
NUM_CLASSES = len(class_names)
print(f"out: {OUT_DIR}\nin:  {DATA_DIR}\n{NUM_CLASSES} classes")

assert set(class_names) == set(list_classes(val_dir)), \
    "train/ and val/ hold different class names — index shifts would silently mislabel everything"
assert set(class_names) == set(list_classes(test_dir)), \
    "train/ and test/ hold different class names"

counts = {n: count_images(os.path.join(train_dir, n)) for n in class_names}
sizes = sorted(counts.values())
print(f"train per class: min {sizes[0]}, median {sizes[len(sizes) // 2]}, max {sizes[-1]}")
thin = [n for n, c in counts.items() if c < 15]
if thin:
    print("!! under 15 train images to low for training:", thin, "— these will make confident wrong predictions")

for name, d in (("val", val_dir), ("test", test_dir)):
    print(f"{name}: {sum(count_images(os.path.join(d, n)) for n in class_names)} images")

## GENERATORS
- The model takes float32 pixels in [0, 255]: EfficientNet's stem
- (Rescaling(1/255) + Normalization) is inside the graph, so nothing out here scales.
- efficientnet.preprocess_input is a no-op and must not be passed to
- flow_from_directory() — it has no such keyword (TypeError on TF >= 2.16 / Keras 3).

In [ ]:
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    rotation_range=AUGMENTATION["rotation"],
    zoom_range=AUGMENTATION["zoom"],
    horizontal_flip=AUGMENTATION["horizontal_flip"],
    brightness_range=(1 - AUGMENTATION["brightness"], 1 + AUGMENTATION["brightness"]),
    contrast_range=(1 - AUGMENTATION["contrast"], 1 + AUGMENTATION["contrast"]),
)
eval_datagen = tf.keras.preprocessing.image.ImageDataGenerator()


def gen(d, g, shuffle=True, seed=None):
    return g.flow_from_directory(
        d, target_size=INPUT_SHAPE[:2], batch_size=BATCH_SIZE,
        class_mode="categorical", shuffle=shuffle, seed=seed,
    )


train_generator = gen(train_dir, train_datagen, shuffle=True, seed=SEED)
val_generator = gen(val_dir, eval_datagen, shuffle=False, seed=SEED)
test_generator = gen(test_dir, eval_datagen, shuffle=False)

CLASS_TO_ID = {n: i for i, n in enumerate(class_names)}
ID_TO_CLASS = {i: class_to_scientific(n) for n, i in CLASS_TO_ID.items()}

assert len(CLASS_TO_ID) == NUM_CLASSES
assert train_generator.num_classes == NUM_CLASSES
assert list(train_generator.class_indices) == class_names, \
    "class_indices order differs from class_names — every label would be shifted"
print("index 0 ->", ID_TO_CLASS[0], "| last ->", ID_TO_CLASS[NUM_CLASSES - 1])

# extension checker
WHITELIST = IMG_EXT 


def find_unsupported(root, names):
    """Files the iterator will never open, grouped by extension. Walks like count_images."""
    by_ext = {}
    for n in names:
        for _, _, files in os.walk(os.path.join(root, n)):
            for f in files:
                if not f.lower().endswith(WHITELIST):
                    by_ext.setdefault(f.rsplit(".", 1)[-1].lower(), []).append(
                        os.path.join(n, f))
    return by_ext


for split, root in (("train", train_dir), ("val", val_dir), ("test", test_dir)):
    found = find_unsupported(root, class_names)
    if found:
        print(f"\n!! {split}: files the loader will silently skip")
        for ext, files in sorted(found.items()):
            print(f"   .{ext:<8} x{len(files):<4} e.g. {files[0]}")
    else:
        print(f"{split}: every file has a supported extension")

assert sum(counts.values()) == train_generator.samples, (
    f"train/ holds {sum(counts.values())} whitelisted images but the generator will load "
    f"only {train_generator.samples} — see the report above.")
print(f"\nwill load — train: {train_generator.samples} | val: {val_generator.samples} | "
      f"test: {test_generator.samples}")

# loading model

In [ ]:

base = tf.keras.applications.EfficientNetB0(
    include_top = False,
    weights = "imagenet",
    input_shape = INPUT_SHAPE,
    pooling = "avg"
)

base.trainable = False

model = tf.keras.Sequential([
    base,
    tf.keras.layers.Dense(512, activation= "relu"),
    tf.keras.layers.Dropout (DROPOUT),
    tf.keras.layers.Dense (NUM_CLASSES, activation="softmax")
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
              loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

# Training loop

In [ ]:
total = sum(counts[n] for n in class_names)
class_weight = {i: total / (NUM_CLASSES * counts[n]) for n, i in CLASS_TO_ID.items()}

history = model.fit(
    train_generator, epochs=EPOCHS, validation_data=val_generator,
    class_weight=class_weight,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=EARLY_STOPPING_PATIENCE,
                                         restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=REDUCE_LR_PATIENCE),
    ],
)


# Model Testing  

In [ ]:
test_loss, test_acc = model.evaluate(test_generator)
print(f"Test accuracy: {test_acc:.4f}")

y_pred = np.argmax(model.predict(test_generator, verbose=0), axis=1)
y_true = test_generator.classes         
labels = [class_to_scientific(n) for n in test_generator.class_indices.keys()]
assert len(labels) == NUM_CLASSES

print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))

cm = confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES))
plt.figure(figsize=(14, 12))
plt.imshow(cm, cmap="bwr"); plt.title("Confusion matrix"); plt.colorbar()
plt.xticks(range(NUM_CLASSES), labels, rotation=90, fontsize=6)
plt.yticks(range(NUM_CLASSES), labels, fontsize=6)
plt.tight_layout(); plt.show()

wrong = np.where(y_pred != y_true)[0]
print(f"Misclassified: {len(wrong)} / {len(y_true)}")
if len(wrong):
    filepaths = test_generator.filepaths
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    for ax, idx in zip(axes.flat, wrong[:10]):
        ax.imshow(plt.imread(filepaths[idx]))   # original file — nothing to undo
        ax.set_title(f"True: {labels[y_true[idx]]}\nPred: {labels[y_pred[idx]]}", fontsize=7)
        ax.axis("off")
    plt.tight_layout(); plt.show()


# Plot training history

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history["loss"], label="Training Loss")
ax1.plot(history.history["val_loss"], label="Validation Loss")
ax1.set_title("Model Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(history.history["accuracy"], label="Training Accuracy")
ax2.plot(history.history["val_accuracy"], label="Validation Accuracy")
ax2.set_title("Model Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()

plt.tight_layout()
plt.show()


# Saving the Model

In [ ]:
model.save(MODEL_PATH)

with open(os.path.join(OUT_DIR, "classes.json"), "w", encoding="utf-8") as f:
    json.dump({str(i): {"scientific": ID_TO_CLASS[i], "trainImages": counts[class_names[i]]}
               for i in range(NUM_CLASSES)},
              f, indent=2, ensure_ascii=False)

print("saved:", sorted(os.listdir(OUT_DIR)))

In [ ]:
with open(os.path.join(OUT_DIR, "classes.json"), encoding="utf-8") as f:
    served = json.load(f)

reloaded = tf.keras.models.load_model(MODEL_PATH)

assert len(served) == reloaded.output_shape[-1], (
    f"classes.json has {len(served)} entries but the model predicts {reloaded.output_shape[-1]} "
    "— different training runs. Every prediction would be confidently wrong.")
assert len(served) == model.output_shape[-1]
assert not any("_" in v["scientific"] for v in served.values()), "underscore translation was skipped"
assert all(v["scientific"] == v["scientific"].strip() for v in served.values())

x, _ = next(iter(eval_datagen.flow_from_directory(test_dir, INPUT_SHAPE[:2], batch_size=1)))
print("sample range:", float(x.min()), "to", float(x.max()),
      "(want 0.0 to 255.0 — the model's stem does the /255, nothing else may)")
print("OK — artifacts consistent")